In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import dask.array as da
import os
import sys
import logging
import seaborn as sns
import datetime
from time import sleep

# Today's date
today = datetime.date.today()
# Format date
date_str = today.strftime("%b%d")

In [ ]:
"""
Per-cell GR variability by (dex_conc, time): STD table for manuscript,
plus a bar chart of the N/C ratio (baseline / peak / final) with t-tests.
- Drops (dex_conc, time) conditions with only one replica.
- Applies a cytoplasmic-area gate BEFORE computing the N/C ratio.
- Reports n, mean, STD for nuc, cyt, N/C ratio.
- Writes a sn-jnl friendly LaTeX table (booktabs).
"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

DATA_PATH = '/Users/ericron/Documents/GitHub/SSIT-workspace/WorkSpace/GR_DUSP1_Model_2025/Data_Aug2025/GR_SSITcellresults_Final_Sep18.csv'
OUT_CSV      = 'GR_variability_by_dose_time.csv'
OUT_TEX      = 'GR_STD_table.tex'
OUT_FIG_BAR  = 'GR_NCratio_barchart.png'

# ------------------------------------------------------------------
# Cytoplasmic-AREA gate (applied to CalcCytoArea), BEFORE the N/C ratio.
# Set explicit bounds, or leave a bound as None to auto-pick it from the
# CalcCytoArea distribution at the percentiles in CYT_AREA_PCT.
# ------------------------------------------------------------------
CYT_AREA_LOW  = 12593     # DUSP1 cyto-area bounds
CYT_AREA_HIGH = 17685

DOSES     = [1, 10, 100]   # nM, ordered for the bar chart
ERRORBAR  = 'sem'          # 'sem' or 'std' for the bar chart error bars


def variability(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    n = len(x)
    if n < 3:
        return pd.Series({'mean': np.nan, 'median': np.nan, 'std': np.nan,
                          'cv': np.nan, 'iqr_over_median': np.nan,
                          'log_std': np.nan, 'n': float(n)})
    mean   = x.mean()
    median = np.median(x)
    q25, q75 = np.percentile(x, [25, 75])
    pos = x[x > 0]
    return pd.Series({
        'mean':            mean,
        'median':          median,
        'std':             x.std(ddof=1),
        'cv':              x.std(ddof=1) / mean   if mean   else np.nan,
        'iqr_over_median': (q75 - q25)  / median if median else np.nan,
        'log_std':         np.log(pos).std(ddof=1) if len(pos) >= 3 else np.nan,
        'n':               float(n),
    })


# load
df = pd.read_csv(DATA_PATH).copy()

# drop conditions with only one replica
n_reps = df.groupby(['dex_conc', 'time'])['replica'].nunique()
keep = n_reps[n_reps > 1].index
df = df.set_index(['dex_conc', 'time']).loc[keep].reset_index()
print(f'Kept {len(keep)} (dex_conc, time) conditions with >=2 replicas.')

# ------------------------------------------------------------------
# cytoplasmic-AREA gate -- BEFORE the N/C ratio is computed
# ------------------------------------------------------------------
n_before = len(df)
df = df[(df['CalcCytoArea'] >= CYT_AREA_LOW) &
        (df['CalcCytoArea'] <= CYT_AREA_HIGH)].copy()
print(f'Cyto-area gate kept {len(df)}/{n_before} cells '
      f'(CalcCytoArea in [{CYT_AREA_LOW}, {CYT_AREA_HIGH}]).')

# per-cell N/C ratio (computed on gated cells)
df['NC_ratio'] = df['normGRnuc'] / df['normGRcyt'].replace(0, np.nan)

# variability table (wide form, all stats)
cols = {
    'NucGR':    'normGRnuc',
    'CytGR':    'normGRcyt',
    'NC_ratio': 'NC_ratio',
}
pieces = []
for label, col in cols.items():
    s = df.groupby(['dex_conc', 'time'])[col].apply(variability).unstack(level=-1)
    s.columns = pd.MultiIndex.from_product([[label], s.columns])
    pieces.append(s)
wide = pd.concat(pieces, axis=1).sort_index()
wide.to_csv(OUT_CSV)

# ------------------------------------------------------------------
# compact view for the manuscript: n, mean + STD per compartment
# ------------------------------------------------------------------
tbl = pd.DataFrame(index=wide.index)
tbl['n']           = wide[('NucGR',    'n')].astype(int)
tbl['NucGR_mean']  = wide[('NucGR',    'mean')]
tbl['NucGR_STD']   = wide[('NucGR',    'std')]
tbl['CytGR_mean']  = wide[('CytGR',    'mean')]
tbl['CytGR_STD']   = wide[('CytGR',    'std')]
tbl['NC_mean']     = wide[('NC_ratio', 'mean')]
tbl['NC_STD']      = wide[('NC_ratio', 'std')]
tbl = tbl.reset_index().sort_values(['dex_conc', 'time'])

print(tbl.round(2).to_string(index=False))

# ------------------------------------------------------------------
# LaTeX table (sn-jnl friendly, booktabs)
# ------------------------------------------------------------------
def fmt_float(x, digits=2):
    return '--' if pd.isna(x) else f'{x:.{digits}f}'

def fmt_int(x):
    return '--' if pd.isna(x) else f'{int(x)}'

body_rows = []
for _, r in tbl.iterrows():
    body_rows.append(' & '.join([
        f'{r["dex_conc"]:g}',
        f'{r["time"]:g}',
        fmt_int(r['n']),
        fmt_float(r['NucGR_mean'], 0),
        fmt_float(r['NucGR_STD'],  0),
        fmt_float(r['CytGR_mean'], 0),
        fmt_float(r['CytGR_STD'],  0),
        fmt_float(r['NC_mean'],    2),
        fmt_float(r['NC_STD'],     2),
    ]) + r' \\')

latex = (
r"""\begin{table}[h]
\centering
\caption{Per-cell variability of GR fluorescence by Dex concentration and stimulation time. Mean intensities are reported in arbitrary units after background correction. SD is the standard deviation across single cells, pooled across biological replicas (conditions with a single replica were excluded). Cells were gated on cytoplasmic area prior to computing the N/C ratio. $n$ is the number of cells per condition.}
\label{tab:GR_variability}
\begin{tabular}{r r r r r r r r r}
\toprule
Dex & Time & & \multicolumn{2}{c}{Nuclear GR} & \multicolumn{2}{c}{Cytoplasmic GR} & \multicolumn{2}{c}{N/C ratio} \\
\cmidrule(lr){4-5} \cmidrule(lr){6-7} \cmidrule(lr){8-9}
(nM) & (min) & $n$ & mean & SD & mean & SD & mean & SD \\
\midrule
""" + '\n'.join(body_rows) + r"""
\bottomrule
\end{tabular}
\end{table}
"""
)

with open(OUT_TEX, 'w') as f:
    f.write(latex)
print(f'\nWrote LaTeX table to {OUT_TEX}')

# ------------------------------------------------------------------
# Bar chart of per-cell N/C ratio:
#   baseline (0 min) | peak & final for each dose
#   mean +/- SEM, Welch's t-tests annotated
# ------------------------------------------------------------------
NC = 'NC_ratio'

def stars(p):
    if p < 1e-4: return '****'
    if p < 1e-3: return '***'
    if p < 1e-2: return '**'
    if p < 5e-2: return '*'
    return 'ns'

def err(vals):
    vals = np.asarray(vals, dtype=float)
    sd = np.nanstd(vals, ddof=1)
    return sd / np.sqrt(np.isfinite(vals).sum()) if ERRORBAR == 'sem' else sd

baseline_time = df['time'].min()
base_vals = df.loc[df['time'] == baseline_time, NC].dropna().values

groups = [{'label': f'{baseline_time:g} min\n(baseline)',
           'vals': base_vals, 'dose': None, 'kind': 'baseline'}]

for dose in DOSES:
    d = df[(df['dex_conc'] == dose) & (df['time'] > baseline_time)]
    if d.empty:
        print(f'[warn] no post-baseline cells for {dose} nM; skipping.')
        continue
    mean_by_t = d.groupby('time')[NC].mean()
    peak_t  = mean_by_t.idxmax()
    final_t = d['time'].max()
    groups.append({'label': f'{dose:g} nM peak\n({peak_t:g} min)',
                   'vals': d.loc[d['time'] == peak_t,  NC].dropna().values,
                   'dose': dose, 'kind': 'peak'})
    groups.append({'label': f'{dose:g} nM final\n({final_t:g} min)',
                   'vals': d.loc[d['time'] == final_t, NC].dropna().values,
                   'dose': dose, 'kind': 'final'})

labels = [g['label'] for g in groups]
means  = [np.nanmean(g['vals']) for g in groups]
errs   = [err(g['vals']) for g in groups]
x = np.arange(len(groups))

# color: baseline grey, one hue per dose
dose_palette = dict(zip(DOSES, sns.color_palette('viridis', n_colors=len(DOSES))))
colors = ['0.6' if g['dose'] is None else dose_palette[g['dose']] for g in groups]

fig, ax = plt.subplots(figsize=(max(7, 1.15 * len(groups)), 5))
bars = ax.bar(x, means, yerr=errs, capsize=4, color=colors,
              edgecolor='black', linewidth=0.6, alpha=0.9)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('Nuclear / cytoplasmic GR ratio')
ax.set_title('GR N/C ratio: baseline vs peak/final response')
ax.margins(y=0.20)

# --- t-tests vs baseline (stars above each treatment bar) ---
print('\nWelch t-tests vs baseline:')
ymax = max(m + e for m, e in zip(means, errs))
for i, g in enumerate(groups[1:], start=1):
    t, p = stats.ttest_ind(g['vals'], base_vals, equal_var=False)
    print(f'  {g["label"].replace(chr(10), " "):<22} '
          f't={t:6.2f}  p={p:.2e}  {stars(p)}')
    ax.text(x[i], means[i] + errs[i] + 0.02 * ymax, stars(p),
            ha='center', va='bottom', fontsize=10)

# --- peak vs final t-tests within each dose (brackets) ---
print('\nWelch t-tests peak vs final (per dose):')
idx_by_dose = {}
for i, g in enumerate(groups):
    if g['dose'] is not None:
        idx_by_dose.setdefault(g['dose'], {})[g['kind']] = i
level = ymax * 1.08
for dose, kinds in idx_by_dose.items():
    if 'peak' in kinds and 'final' in kinds:
        ip, iff = kinds['peak'], kinds['final']
        t, p = stats.ttest_ind(groups[ip]['vals'], groups[iff]['vals'],
                               equal_var=False)
        print(f'  {dose:>3g} nM peak vs final   '
              f't={t:6.2f}  p={p:.2e}  {stars(p)}')
        h = level
        ax.plot([x[ip], x[ip], x[iff], x[iff]],
                [h, h * 1.02, h * 1.02, h], lw=1.0, color='black')
        ax.text((x[ip] + x[iff]) / 2, h * 1.02, stars(p),
                ha='center', va='bottom', fontsize=9)
        level *= 1.12

# --- endpoint (final) comparisons across the three doses (brackets) ---
print('\nWelch t-tests final vs final (across doses):')
final_idx = {g['dose']: i for i, g in enumerate(groups)
             if g['dose'] is not None and g['kind'] == 'final'}
dose_pairs = [(DOSES[a], DOSES[b])
              for a in range(len(DOSES)) for b in range(a + 1, len(DOSES))]
for da, db in dose_pairs:
    if da in final_idx and db in final_idx:
        ia, ib = final_idx[da], final_idx[db]
        t, p = stats.ttest_ind(groups[ia]['vals'], groups[ib]['vals'],
                               equal_var=False)
        print(f'  {da:>3g} vs {db:>3g} nM final  '
              f't={t:6.2f}  p={p:.2e}  {stars(p)}')
        h = level
        ax.plot([x[ia], x[ia], x[ib], x[ib]],
                [h, h * 1.02, h * 1.02, h], lw=1.0, color='black')
        ax.text((x[ia] + x[ib]) / 2, h * 1.02, stars(p),
                ha='center', va='bottom', fontsize=9)
        level *= 1.12

ax.set_ylim(top=level * 1.10)   # make room for the stacked brackets
sns.despine(ax=ax)
fig.tight_layout()
fig.savefig(OUT_FIG_BAR, dpi=600)
print(f'\nWrote bar chart to {OUT_FIG_BAR}')